# Exploración y búsqueda de datos

En este notebook se testea el funcionamiento de `economista.data.search`

Debemos pensar `data.search` como la __API de exploración de catálogos__: sirve para descubrir indicadores, países/códigos ISO y temas antes de usar `data.fetch`

In [ ]:
import pandas as pd

from economista import data

pd.set_option("display.max_colwidth", None)


El contrato es el siguiente:

```python
data.search(
    query=None,
    *,
    source=None,
    dataset=None,
    kind="all",
    limit=25,
    as_frame=True,
    **filters,
)
```

Y los parámetros hacen esto:

---
`query` es búsqueda abierta lexical. Busca texto dentro de los catálogos oficiales: nombres, claves, notas, regiones, temas, códigos ISO, etc.:

```python
data.search("GDP")
data.search("external debt")
data.search("Mexico")
data.search("MEX")
```
---
`source` es para especificar conector. Si se omite, busca en todos los conectores disponibles (al día de hoy, 11 de jul 2026, sólo está disponible WB/WDI):

```python
data.search("GDP", source="world_bank")
data.search("GDP")  # búsqueda global en conectores registrados
```
---
`kind` define qué catálogo quieres explorar. Si se omite, se busca en todos los catálogos ("all"):

```python
kind="all"         # indicadores, países/geos y temas
kind="indicators"  # indicadores económicos
kind="geos"        # países, regiones/agregados, códigos ISO
kind="topics"      # temas/categorías
```
---
`limit` controla el número de resultados que devuelve la consulta. Si se omite, se devuelven 25 resultados:

```python
data.search("GDP", limit=25)     # default
data.search("GDP", limit=10)
data.search("GDP", limit=None)   # todos los resultados disponibles
```
---
`as_frame` define si quieres recibir los resultados como un `pandas.DataFrame` (ideal para notebooks como este) o bien como una lista nativa de Python. Si se omite, devuelve DataFrame.

```python
rows = data.search("GDP", kind="indicators", as_frame=False) # Devuelve como lista de Python
rows = data.search("GDP", kind="indicators") # Devuelve pandas.DataFrame
```
---

Dicho esto, podemos hacer algunos ejemplos.

Buscar indicadores relacionados con el PIB:

In [ ]:
gdp_search = data.search(
    query="gdp",
    kind="indicators"
)

#ver los primeros 5 resultados
gdp_search.head()

Resultados de búsqueda como lista de Python y con más resultados (40)

In [ ]:
gdp_search_list = data.search(
    query="gdp",
    kind="indicators",
    as_frame=False,
    limit=40
)

print(len(gdp_search_list))

for _ in gdp_search_list:
    print(_)

Otro ejemplo puede ser buscar un indicador exacto por su clave:

In [ ]:
# inversión extranjera directa
data.search(
    query="BM.KLT.DINV.WD.GD.ZS",
    source="world_bank",
)

Buscar indicadores de innovación:

In [ ]:
# Buscaremos en todas las fuentes y catálogos,
# sin definir parámetros `source` ni `kind`.
# Se pasa el único parámetro obligatorio: `query`.
external_debt_search = data.search("innovation")

external_debt_search


__Nota:__ El texto no es legible en columnas como `name` o `source_note`. Para corregir esto, se puede configurar pandas de forma global para ver todo el contenido de las columnas. Es opcional:

In [ ]:
external_debt_search


Buscar México, Argentina, Marruecos e Italia para obtener códigos ISO:

In [ ]:
mexico_iso = data.search(
    query="México",
    kind="geos"
)

marruecos_iso = data.search(
    query="Morocco",
    kind="geos"
)

italia_iso = data.search(
    query="Italy",
    kind="geos"
)

paises_iso = pd.concat([mexico_iso, marruecos_iso, italia_iso])
paises_iso

In [ ]:
# TODOS LOS PAÍSES
data.search(
    query="",
    kind="geos",
    limit=None,
)

También se puede lo inverso: buscar por ISO2 o ISO3:

In [ ]:
data.search("MX", source="world_bank", kind="geos")

In [ ]:
data.search("MEX", source="world_bank", kind="geos")

In [ ]:
# Quiero saber qué país es "AD"
data.search("AD", source="world_bank", kind="geos")

Buscar países o agregados de Europa:

In [ ]:
# aquí no se pasa parámetro `query`
europa_search = data.search(
    source="world_bank",
    kind="geos",
    region="Europe",
)

europa_search

Buscar temas relacionados con Deuda:

In [ ]:
data.search(
    "debt",
    kind="topics"
)

O temas relacionados con agricultura:

In [ ]:
data.search(
    "agriculture",
    kind="topics"
)

O si andas falto de ideas para temas, la query en blanco, con todos los resultados y como lista de objetos de Python:

In [ ]:
data.search(
    "",
    kind="topics",
    limit=None,
    as_frame=False
)

Buscar en todos los catálogos a la vez (con parámetros explícitos):

In [ ]:
todo_deuda = data.search(
    "external debt",
    kind="all",
    limit=None
)

todo_deuda

## Flujo típico de búsqueda antes de `data.fetch`

Primero, hacemos búsqueda de indicadores:

In [ ]:
# Se pueden concatenar métodos de pandas al output de data.search.
# Por ejemplo, sample().

# 20 indicadores al azar
data.search("", kind="indicators", limit=None).sample(20)


Elegimos Importación de bienes y servicios (constante 2025): `Imports of goods and services (constant 2015 US$)` y su ID de World Bank: `NE.IMP.GNFS.KD`

`data.search` ha cumplido su cometido: explorar y encontrar indicadores de interés.

Ahora, buscamos el ISO del país de interés:

In [ ]:
data.search(
    "Chile",
    kind="geos"
)

Ahora ya sabemos que para Chile, su ISO2 es `CL` e ISO3 es `CHL`.

Procedemos a extraer los datos con `data.fetch`:

In [ ]:
indicador="NE.IMP.GNFS.KD"

chile_imports = data.fetch(
    source="world_bank",
    dataset="wdi",
    indicator=indicador,
    geo="CL",
    start=2000,
    end=2020
)

df = chile_imports.to_pandas()
df.head()

Para ver los metadatos del indicador extraído:

In [ ]:
chile_imports.metadata.to_dict()

__Nota importante__: `data.search` todavía no verifica cobertura cruzada completa. Es decir:
```python
data.search("external debt", kind="geos")
```
No significa "países con datos de deuda externa". Significa "buscar `external debt` dentro del catálogo `geos`, lo cual puede no ser útil en todos los casos. Para eso, lo correcto hoy es buscar primero el indicador y luego usar `data.fetch` con países concretos. La cobertura indicador-país queda como futura característica.

In [ ]:
external_debt_search.loc[0, "source_note"]